# HyperSSM-1.58 Training on Google Colab
**OpenAI Parameter Golf Challenge**

This notebook trains the HyperSSM-1.58 model on Colab GPU.

**Steps:**
1. Setup environment & clone repo
2. Download dataset
3. Train model
4. Evaluate & save artifact
5. Push results back to GitHub

## 0. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Install dependencies & clone repo

In [ ]:
!pip install -q sentencepiece huggingface-hub datasets tqdm

import os
os.chdir('/content')

# Clone your fork
if not os.path.exists('parameter-golf'):
    !git clone https://github.com/ironbyte-rgb/parameter-golf.git
    !cd parameter-golf && git checkout hyperssm-1.58-submission
else:
    !cd parameter-golf && git pull && git checkout hyperssm-1.58-submission

os.chdir('/content/parameter-golf')
print("\nRepo ready!")
!ls records/track_10min_16mb/2026-03-23_HyperSSM158/

## 2. Download dataset

In [ ]:
# Download validation + training shards
# Use --train-shards 10 for a good balance (10 shards = ~1B tokens)
# Use --train-shards 80 for full dataset (8B tokens) if you have time
!python data/cached_challenge_fineweb.py --variant sp1024 --train-shards 10

!ls data/datasets/fineweb10B_sp1024/
!ls data/tokenizers/

## 3. Quick sanity check (optional)

In [ ]:
# Quick 30-step test to make sure everything works
!TEST_MODE=1 ITERATIONS=30 TRAIN_LOG_EVERY=10 VAL_LOSS_EVERY=0 \
  python records/track_10min_16mb/2026-03-23_HyperSSM158/train_gpt.py

## 4. Full Training Run

On Colab Pro with a single GPU (A100/V100/T4), we use `nproc_per_node=1`.
Adjust `MAX_WALLCLOCK_SECONDS` based on your Colab session limits.

In [ ]:
# Copy train script to repo root (torchrun expects it there or use full path)
!cp records/track_10min_16mb/2026-03-23_HyperSSM158/train_gpt.py train_hyperssm.py

# Full training run on single GPU
# - 10 min wallclock (adjust if needed)
# - Logs every 50 steps, validates every 500 steps
!RUN_ID=hyperssm158_colab \
  DATA_PATH=./data/datasets/fineweb10B_sp1024 \
  TOKENIZER_PATH=./data/tokenizers/fineweb_1024_bpe.model \
  VOCAB_SIZE=1024 \
  MAX_WALLCLOCK_SECONDS=600 \
  TRAIN_LOG_EVERY=50 \
  VAL_LOSS_EVERY=500 \
  TRAIN_BATCH_TOKENS=65536 \
  torchrun --standalone --nproc_per_node=1 train_hyperssm.py 2>&1 | tee train_hyperssm.log

## 5. Check results

In [ ]:
import os

# Check artifact size
if os.path.exists('final_model.int8.ptz'):
    size = os.path.getsize('final_model.int8.ptz')
    code_size = os.path.getsize('train_hyperssm.py')
    total = size + code_size
    print(f"Model artifact: {size:,} bytes ({size/1e6:.2f} MB)")
    print(f"Code: {code_size:,} bytes")
    print(f"Total: {total:,} bytes ({total/1e6:.2f} MB)")
    print(f"Under 16MB: {total <= 16_000_000}")
else:
    print("No artifact found - training may not have completed")

# Show final metrics from log
print("\n--- Final metrics ---")
!grep -E 'final_int8_zlib|val_bpb|stopping' train_hyperssm.log 2>/dev/null || echo 'No log found'

# Show last few training steps
print("\n--- Last training steps ---")
!tail -20 train_hyperssm.log 2>/dev/null || echo 'No log found'

## 6. Save results & push to GitHub

In [ ]:
# Copy training log to submission folder
!cp train_hyperssm.log records/track_10min_16mb/2026-03-23_HyperSSM158/train.log 2>/dev/null

# Extract final val_bpb from log and update submission.json
import json, re

log_path = 'train_hyperssm.log'
submission_path = 'records/track_10min_16mb/2026-03-23_HyperSSM158/submission.json'

val_bpb = 0.0
val_loss = 0.0
artifact_size = 0

if os.path.exists(log_path):
    with open(log_path) as f:
        log_text = f.read()
    
    # Find final roundtrip metrics
    m = re.search(r'final_int8_zlib_roundtrip_exact val_loss:([\d.]+) val_bpb:([\d.]+)', log_text)
    if m:
        val_loss = float(m.group(1))
        val_bpb = float(m.group(2))
    
    m = re.search(r'Total submission size int8\+zlib: (\d+) bytes', log_text)
    if m:
        artifact_size = int(m.group(1))

if val_bpb > 0:
    submission = {
        "author": "ironbyte-rgb",
        "github_id": "ironbyte-rgb",
        "name": "HyperSSM-1.58",
        "blurb": f"Selective SSM + hypernetwork weight generation + 1.58-bit ternary. val_bpb={val_bpb:.4f}",
        "date": "2026-03-23",
        "val_loss": val_loss,
        "val_bpb": val_bpb,
        "bytes_total": artifact_size,
        "bytes_code": os.path.getsize('records/track_10min_16mb/2026-03-23_HyperSSM158/train_gpt.py')
    }
    with open(submission_path, 'w') as f:
        json.dump(submission, f, indent=2)
    print(f"Updated submission.json: val_bpb={val_bpb:.4f}")
else:
    print("Could not extract val_bpb from log")

In [ ]:
# Authenticate with GitHub (run this cell and follow the prompts)
# Option 1: Use a Personal Access Token
# Go to https://github.com/settings/tokens -> Generate new token (classic)
# Select 'repo' scope -> Generate -> Copy token

from getpass import getpass
token = getpass("Enter your GitHub Personal Access Token: ")

!git remote set-url origin https://{token}@github.com/ironbyte-rgb/parameter-golf.git
print("GitHub auth configured!")

In [ ]:
# Commit and push results
!git add records/track_10min_16mb/2026-03-23_HyperSSM158/
!git status
!git commit -m "Add training results for HyperSSM-1.58 (Colab run)"
!git push origin hyperssm-1.58-submission

## 7. Download artifacts locally (optional)

In [ ]:
# Download the trained model and log
from google.colab import files

if os.path.exists('final_model.int8.ptz'):
    files.download('final_model.int8.ptz')
if os.path.exists('train_hyperssm.log'):
    files.download('train_hyperssm.log')